# Notebook 4: Feature Engineering & Time-Series Signals
### Project: Retail Sales Demand Forecasting & Inventory Optimization
**Objective**: Construct date-based calendar features, net revenue calculations, and strictly non-lookahead historical demand features (Lag_1, Lag_7, Lag_30, Rolling_Mean_7, Rolling_Mean_30).

---
### 1. Feature Engineering Principles & Leakage Prevention
To prevent data leakage in time-series forecasting, historical demand features must NEVER include the target observation itself:
- `Lag_1` = `shift(1)`
- `Rolling_Mean_7` = `shift(1).rolling(7).mean()`
Calculations must be partitioned per `(Store_ID, Product_ID)` series.


In [ ]:
import sys
import os
import pandas as pd

sys.path.insert(0, os.path.abspath('..'))
from src.feature_engineering import FeatureEngineer

clean_df = pd.read_csv('../dataset/processed/cleaned_sales.csv')
fe = FeatureEngineer()
feat_df = fe.transform(clean_df)

print("Engineered DataFrame shape:", feat_df.shape)
print("Features created:", list(feat_df.columns))
feat_df[['Date', 'Store_ID', 'Product_ID', 'Units_Sold', 'Lag_1', 'Lag_7', 'Rolling_Mean_7', 'Rolling_Mean_30']].head(10)


### Feature Verification
Notice how `Lag_1` for each store-product group strictly mirrors the previous day's `Units_Sold`, ensuring zero lookahead bias. The rolling windows provide smoothed moving averages of recent demand velocity.


In [ ]:
# Inspect feature correlation with Units_Sold
numeric_cols = ['Units_Sold', 'Unit_Price', 'Discount', 'Promotion', 'Holiday', 
                'Day_of_Week', 'Is_Weekend', 'Lag_1', 'Lag_7', 'Rolling_Mean_7', 'Rolling_Mean_30']
corr = feat_df[numeric_cols].corr()

import matplotlib.pyplot as plt
import seaborn as sns
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', cbar=True)
plt.title('Feature Correlation Matrix with Units_Sold', fontweight='bold')
plt.show()


### Analytical Insight from Correlation Matrix
- `Rolling_Mean_7`, `Rolling_Mean_30`, and `Lag_1` have strong positive correlations (>0.85) with `Units_Sold`, confirming their predictive power.
- `Promotion` and `Is_Weekend` demonstrate strong positive correlation with demand spikes.

---
### Conclusion
We have engineered 24 comprehensive features with zero lookahead leakage, providing a high-dimensional input representation for our ML regressors.
